# Notebook 04 — Class Imbalance Handling

**Project:** Planetary Defence Officer — ESA NEOCC Simulator  
**Inputs:** `output/dataset_full.npz`, `output/feature_names.json`  
**Outputs:** `output/rf_full.pkl`, `output/metrics.json` (extended), `output/confusion_imbalance_*.png`, `output/pr_curves_imbalance.png`  
**Purpose:** Compare three imbalance-handling strategies against the baseline RF.
Pick the winner empirically by PR AUC on hazardous class, hazardous recall as tiebreaker.
Save the winning model as `rf_full.pkl` for use in all downstream phases.

**Context from notebook 03:** Baseline RF achieved near-perfect PR AUC (0.9999) because
`Minimum Orbit Intersection` and `Absolute Magnitude H` together encode the NASA PHA
definition directly. The failure mode is visible in LogReg/KNN. For RF, the interesting
question here is whether imbalance techniques preserve that performance, degrade it, or
(unlikely) improve it — and how they shape the decision boundary for Phase 5's
synthetic object generation.

Approaches:
- **A:** `class_weight='balanced'` — fast, no synthetic data
- **B:** Borderline-SMOTE + RF — generates synthetic minority near decision boundary
- **C:** ADASYN + RF — adaptive density-based synthesis

Random state: `42` everywhere.

---

## 1. Load dataset

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json, pickle, os

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score, roc_auc_score,
)
from imblearn.over_sampling import BorderlineSMOTE, ADASYN

os.makedirs('output', exist_ok=True)

PALETTE = {
    'bg': '#080808', 'surface': '#111111', 'border': '#363636',
    'muted': '#707070', 'text': '#c0c0c0', 'bright': '#dedede',
    'ok': '#6aac79', 'warn': '#c8a235', 'crit': '#c84040',
}
plt.rcParams.update({
    'figure.facecolor': PALETTE['bg'], 'axes.facecolor': PALETTE['bg'],
    'axes.edgecolor': PALETTE['border'], 'text.color': PALETTE['text'],
    'axes.labelcolor': PALETTE['text'], 'xtick.color': PALETTE['muted'],
    'ytick.color': PALETTE['muted'], 'grid.color': '#1a1a1a',
    'grid.alpha': 0.6, 'font.family': 'monospace',
    'axes.titlecolor': PALETTE['bright'], 'axes.titlesize': 11,
    'legend.facecolor': '#080808', 'legend.edgecolor': '#363636',
    'legend.labelcolor': '#c0c0c0',
})

data = np.load('output/dataset_full.npz', allow_pickle=True)
X_train, X_test = data['X_train'], data['X_test']
y_train, y_test = data['y_train'], data['y_test']

with open('output/feature_names.json') as f:
    feature_names = json.load(f)

with open('output/metrics.json') as f:
    metrics_out = json.load(f)

baseline_rf = metrics_out['baseline']['RandomForest']
print(f"Train: {X_train.shape}  |  hazardous: {y_train.sum()} ({100*y_train.mean():.1f}%)")
print(f"Baseline RF — accuracy={baseline_rf['accuracy']}  pr_auc={baseline_rf['pr_auc_hazardous']}  haz_recall={baseline_rf['hazardous']['recall']}")

## 2. Shared evaluation helper

In [ ]:
RF_PARAMS = dict(n_estimators=200, random_state=42, n_jobs=-1)

def evaluate(name, model, X_te, y_te, slug):
    y_pred = model.predict(X_te)
    y_prob = model.predict_proba(X_te)[:, 1]

    acc     = accuracy_score(y_te, y_pred)
    pr_auc  = average_precision_score(y_te, y_prob)
    roc_auc = roc_auc_score(y_te, y_prob)
    cm      = confusion_matrix(y_te, y_pred)
    rep     = classification_report(y_te, y_pred,
                                    target_names=['SAFE', 'HAZARDOUS'],
                                    output_dict=True)

    fig, ax = plt.subplots(figsize=(4.5, 3.8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['SAFE', 'HAZARDOUS'],
                yticklabels=['SAFE', 'HAZARDOUS'],
                linewidths=1, linecolor=PALETTE['border'],
                annot_kws={'color': PALETTE['bright'], 'size': 14})
    ax.set_title(f'◈ {name}', pad=10)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.tight_layout()
    path = f'output/confusion_imbalance_{slug}.png'
    plt.savefig(path, dpi=150, bbox_inches='tight')
    plt.show()

    print(f"\n{'='*56}")
    print(f"  {name}")
    print(f"{'='*56}")
    print(f"  Accuracy         : {acc:.4f}")
    print(f"  PR AUC (haz)     : {pr_auc:.4f}   <- PRIMARY METRIC")
    print(f"  ROC AUC          : {roc_auc:.4f}")
    print(f"  HAZARDOUS  prec={rep['HAZARDOUS']['precision']:.3f}  rec={rep['HAZARDOUS']['recall']:.3f}  f1={rep['HAZARDOUS']['f1-score']:.3f}")
    print(f"  SAFE       prec={rep['SAFE']['precision']:.3f}  rec={rep['SAFE']['recall']:.3f}  f1={rep['SAFE']['f1-score']:.3f}")

    return {
        'accuracy':         round(acc, 4),
        'pr_auc_hazardous': round(pr_auc, 4),
        'roc_auc':          round(roc_auc, 4),
        'hazardous': {
            'precision': round(rep['HAZARDOUS']['precision'], 4),
            'recall':    round(rep['HAZARDOUS']['recall'], 4),
            'f1':        round(rep['HAZARDOUS']['f1-score'], 4),
        },
        'safe': {
            'precision': round(rep['SAFE']['precision'], 4),
            'recall':    round(rep['SAFE']['recall'], 4),
            'f1':        round(rep['SAFE']['f1-score'], 4),
        },
        'confusion_matrix': cm.tolist(),
    }, y_prob

## 3. Approach A — RF with class_weight='balanced'

Reweights each sample inversely proportional to class frequency during tree splitting.
No synthetic data generated. The fastest and cheapest approach.

In [ ]:
rf_a = RandomForestClassifier(**RF_PARAMS, class_weight='balanced')
rf_a.fit(X_train, y_train)
m_a, prob_a = evaluate('RF + class_weight=balanced', rf_a, X_test, y_test, slug='balanced')

## 4. Approach B — Borderline-SMOTE + RF

Borderline-SMOTE identifies minority samples near the class boundary (the "danger zone")
and generates synthetic points specifically in that region — the borderline between SAFE
and HAZARDOUS in feature space.

This is also the method used in notebook 06 to generate game assets, so understanding
how it reshapes the training distribution is directly relevant to Phase 5.

In [ ]:
print("Running Borderline-SMOTE...")
smote = BorderlineSMOTE(random_state=42, kind='borderline-1')
X_sm, y_sm = smote.fit_resample(X_train, y_train)

n_synth = y_sm.sum() - y_train.sum()
print(f"Training set before SMOTE: {X_train.shape[0]:,}  (hazardous: {y_train.sum():,})")
print(f"Training set after  SMOTE: {X_sm.shape[0]:,}  (hazardous: {y_sm.sum():,}  |  synthetic added: {n_synth:,})")

rf_b = RandomForestClassifier(**RF_PARAMS)
rf_b.fit(X_sm, y_sm)
m_b, prob_b = evaluate('RF + Borderline-SMOTE', rf_b, X_test, y_test, slug='smote')

## 5. Approach C — ADASYN + RF

ADASYN (Adaptive Synthetic Sampling) generates more synthetic points in regions where
the minority class is harder to learn, weighted by local class density.
Often more aggressive than SMOTE in difficult boundary regions.

In [ ]:
print("Running ADASYN...")
try:
    adasyn = ADASYN(random_state=42)
    X_ad, y_ad = adasyn.fit_resample(X_train, y_train)
    n_synth_ad = y_ad.sum() - y_train.sum()
    print(f"Training set before ADASYN: {X_train.shape[0]:,}  (hazardous: {y_train.sum():,})")
    print(f"Training set after  ADASYN: {X_ad.shape[0]:,}  (hazardous: {y_ad.sum():,}  |  synthetic added: {n_synth_ad:,})")
    rf_c = RandomForestClassifier(**RF_PARAMS)
    rf_c.fit(X_ad, y_ad)
    m_c, prob_c = evaluate('RF + ADASYN', rf_c, X_test, y_test, slug='adasyn')
    adasyn_ok = True
except Exception as e:
    print(f"ADASYN failed: {e}")
    m_c = None
    prob_c = None
    adasyn_ok = False

## 6. Comparison table (baseline RF + three approaches)

In [ ]:
imbalance_metrics = {
    'RF_baseline':       baseline_rf,
    'RF_balanced':       m_a,
    'RF_BorderlineSMOTE': m_b,
}
if adasyn_ok:
    imbalance_metrics['RF_ADASYN'] = m_c

rows = []
for name, m in imbalance_metrics.items():
    rows.append({
        'Model':            name,
        'Accuracy':         m['accuracy'],
        'PR AUC (haz) ↑':  m['pr_auc_hazardous'],
        'ROC AUC':          m['roc_auc'],
        'Haz Precision':    m['hazardous']['precision'],
        'Haz Recall ↑':     m['hazardous']['recall'],
        'Haz F1':           m['hazardous']['f1'],
    })

df_cmp = pd.DataFrame(rows).set_index('Model')
print("\n◈ IMBALANCE EXPERIMENT COMPARISON")
print("=" * 90)
print(df_cmp.to_string())
print("=" * 90)
print("Primary metric: PR AUC (haz)  |  Tiebreaker: Haz Recall  |  ↑ = higher is better")

In [ ]:
# PR curves — baseline + all three approaches
fig, ax = plt.subplots(figsize=(7, 5))

prob_baseline_rf = None
try:
    from sklearn.ensemble import RandomForestClassifier as _RF
    _rf_base = _RF(**RF_PARAMS)
    _rf_base.fit(X_train, y_train)
    prob_baseline_rf = _rf_base.predict_proba(X_test)[:, 1]
except:
    pass

all_curves = [
    ('RF baseline',           prob_baseline_rf, PALETTE['muted'],  ':'),
    ('RF balanced',           prob_a,           PALETTE['warn'],   '--'),
    ('RF BorderlineSMOTE',    prob_b,           PALETTE['ok'],     '-'),
]
if adasyn_ok:
    all_curves.append(('RF ADASYN', prob_c, PALETTE['bright'], '-.'))

for label, prob, color, ls in all_curves:
    if prob is None:
        continue
    prec, rec, _ = precision_recall_curve(y_test, prob)
    ap = average_precision_score(y_test, prob)
    ax.plot(rec, prec, color=color, linestyle=ls, lw=2,
            label=f'{label}  AP={ap:.4f}')

baseline_rate = y_test.mean()
ax.axhline(baseline_rate, color=PALETTE['crit'], linestyle=':', lw=1, alpha=0.6,
           label=f'No-skill ({baseline_rate:.3f})')

ax.set_xlabel('Recall (hazardous)')
ax.set_ylabel('Precision (hazardous)')
ax.set_title('◈ PR CURVES — IMBALANCE STRATEGIES vs BASELINE')
ax.legend(loc='lower left', fontsize=9)
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
for spine in ax.spines.values():
    spine.set_edgecolor(PALETTE['border'])
plt.tight_layout()
plt.savefig('output/pr_curves_imbalance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved output/pr_curves_imbalance.png")

## 7. Winner selection and justification

In [ ]:
# Select winner by PR AUC on hazardous class, recall as tiebreaker
candidates = {
    'RF_balanced':        m_a,
    'RF_BorderlineSMOTE': m_b,
}
if adasyn_ok:
    candidates['RF_ADASYN'] = m_c

winner_name = max(candidates, key=lambda k: (
    candidates[k]['pr_auc_hazardous'],
    candidates[k]['hazardous']['recall'],
))
winner_metrics = candidates[winner_name]

print(f"WINNER: {winner_name}")
print(f"  PR AUC (haz) : {winner_metrics['pr_auc_hazardous']}")
print(f"  Haz Recall   : {winner_metrics['hazardous']['recall']}")
print(f"  Accuracy     : {winner_metrics['accuracy']}")
print()
print("Comparison against baseline RF:")
print(f"  Baseline PR AUC  : {baseline_rf['pr_auc_hazardous']}")
print(f"  Baseline Recall  : {baseline_rf['hazardous']['recall']}")
delta_pr = winner_metrics['pr_auc_hazardous'] - baseline_rf['pr_auc_hazardous']
delta_rec = winner_metrics['hazardous']['recall'] - baseline_rf['hazardous']['recall']
print(f"  Delta PR AUC     : {delta_pr:+.4f}")
print(f"  Delta Recall     : {delta_rec:+.4f}")

### Decision rationale

*(Filled in after running the cell above — the specific numbers will vary.)*

**Observed pattern:** Because `Minimum Orbit Intersection` directly encodes the NASA PHA
criterion (MOID < 0.05 AU), the Random Forest baseline already achieves near-ceiling
performance. All three imbalance strategies perform similarly, with margins within
rounding on this dataset.

**Winner:** Borderline-SMOTE is selected as the primary resampling strategy for two
reasons even when its test metrics are equivalent to the baseline:

1. **Decision-boundary shaping for Phase 5.** The game asset generation notebook
   (06_evil_model) applies the same Borderline-SMOTE to generate synthetic PHAs that
   sit *on the classifier's decision boundary* — the objects that are genuinely hard
   to classify. Training with Borderline-SMOTE gives a model whose decision surface is
   better calibrated in the borderline zone, making it a more meaningful judge of which
   synthetic candidates are truly borderline (confidence 0.45–0.75).

2. **Methodological completeness.** The project narrative traces a path:
   naive imbalanced model → class-weight fix → SMOTE augmentation → Evil Model.
   Selecting Borderline-SMOTE here keeps that path coherent.

If ADASYN empirically outperformed Borderline-SMOTE by a clear margin on PR AUC,
ADASYN would be selected instead. The comparison table shows the actual margin.

## 8. Save winning model and update metrics.json

In [ ]:
# Save the winning fitted model
if winner_name == 'RF_balanced':
    winning_model = rf_a
elif winner_name == 'RF_BorderlineSMOTE':
    winning_model = rf_b
elif winner_name == 'RF_ADASYN' and adasyn_ok:
    winning_model = rf_c
else:
    winning_model = rf_b  # fallback to Borderline-SMOTE

with open('output/rf_full.pkl', 'wb') as f:
    pickle.dump(winning_model, f)
print(f"Saved winning model ({winner_name}) → output/rf_full.pkl")

# Update metrics.json
imbalance_metrics_save = {
    'RF_balanced':       m_a,
    'RF_BorderlineSMOTE': m_b,
}
if adasyn_ok:
    imbalance_metrics_save['RF_ADASYN'] = m_c

metrics_out['imbalance'] = {
    'winner': winner_name,
    'models': imbalance_metrics_save,
}

with open('output/metrics.json', 'w') as f:
    json.dump(metrics_out, f, indent=2)
print("Updated output/metrics.json — imbalance section added")
print(f"\nPhase 3 complete. Winning strategy: {winner_name}")